# Sequence Models — RNN, LSTM & GRU
## Hands-on Lab: Next-Step Sequence Prediction

### Learning Objectives
By the end of this notebook, you should be able to:

- Prepare sequential data using **sliding windows**.
- Explain the Keras input shape: **samples × timesteps × features**.
- Build and train `SimpleRNN`, `LSTM`, and `GRU`.
- Compare model performance and parameter count.
- Understand when `return_sequences=True` is needed.
- Complete coding and reasoning tasks independently.

### Pipeline
**Raw Sequence → Sliding Windows → 3D Input → Sequence Model → Prediction → Evaluation**


## 1. Import Libraries

In [ ]:
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


## 2. Create Sequential Data

We will use a **sine wave** as a simple sequence.

Our task:

> Given the previous `N` values, predict the next value.


In [ ]:
n_points = 1000
x_axis = np.linspace(0, 40, n_points)
series = np.sin(x_axis)

print("Series shape:", series.shape)
print("First 10 values:", series[:10])


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(series)
plt.title("Raw Sequential Data — Sine Wave")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.show()


## 3. Chronological Train/Test Split

Sequence order matters, so we keep the original order.

- First 80% → training
- Last 20% → testing


In [ ]:
split_index = int(len(series) * 0.80)

train_series = series[:split_index]
test_series = series[split_index:]

print("Training points:", len(train_series))
print("Test points:", len(test_series))


## 4. Sliding Windows

If `seq_length = 10`:

`[x1 ... x10] → x11`

then:

`[x2 ... x11] → x12`

This converts the raw sequence into supervised learning samples.


In [ ]:
def create_sequences(data, seq_length=10):
    X, y = [], []

    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])

    X = np.array(X)
    y = np.array(y)

    # Keras sequence layers expect 3D input:
    # samples × timesteps × features
    X = X.reshape(X.shape[0], X.shape[1], 1)

    return X, y


In [ ]:
SEQ_LENGTH = 10

X_train, y_train = create_sequences(train_series, SEQ_LENGTH)
X_test, y_test = create_sequences(test_series, SEQ_LENGTH)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


### Understanding the Shape

For example:

`X_train.shape = (790, 10, 1)`

means:

- `790` samples
- `10` timesteps per sample
- `1` feature at each timestep


In [ ]:
sample_id = 0

print("Input sequence:")
print(X_train[sample_id].flatten())

print("\nTarget:")
print(y_train[sample_id])


## 5. Helper Function for Training

We will train each model under similar conditions so the comparison is fair.


In [ ]:
def train_and_evaluate(model, X_train, y_train, X_test, y_test, epochs=50):
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    start = time.time()

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.20,
        epochs=epochs,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    training_time = time.time() - start

    predictions = model.predict(X_test, verbose=0).flatten()

    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)

    return history, predictions, mse, mae, training_time


# Part A — Simple RNN

A basic RNN processes the sequence step by step and carries a **hidden state** forward.


In [ ]:
rnn_model = Sequential([
    keras.Input(shape=(SEQ_LENGTH, 1)),
    SimpleRNN(32),
    Dense(1)
])

rnn_model.summary()


In [ ]:
rnn_history, rnn_pred, rnn_mse, rnn_mae, rnn_time = train_and_evaluate(
    rnn_model,
    X_train, y_train,
    X_test, y_test
)

print("RNN MSE:", round(rnn_mse, 6))
print("RNN MAE:", round(rnn_mae, 6))
print("Training time:", round(rnn_time, 2), "seconds")


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(y_test[:120], label="Actual")
plt.plot(rnn_pred[:120], label="RNN Prediction")
plt.title("Simple RNN — Actual vs Predicted")
plt.legend()
plt.show()


# Part B — LSTM

LSTM improves memory control using:

- Forget Gate
- Input Gate
- Output Gate
- Cell State


In [ ]:
lstm_model = Sequential([
    keras.Input(shape=(SEQ_LENGTH, 1)),
    LSTM(32),
    Dense(1)
])

lstm_model.summary()


In [ ]:
lstm_history, lstm_pred, lstm_mse, lstm_mae, lstm_time = train_and_evaluate(
    lstm_model,
    X_train, y_train,
    X_test, y_test
)

print("LSTM MSE:", round(lstm_mse, 6))
print("LSTM MAE:", round(lstm_mae, 6))
print("Training time:", round(lstm_time, 2), "seconds")


# Part C — GRU

GRU is a simpler gated recurrent model with fewer gates and usually fewer parameters than LSTM.


In [ ]:
gru_model = Sequential([
    keras.Input(shape=(SEQ_LENGTH, 1)),
    GRU(32),
    Dense(1)
])

gru_model.summary()


In [ ]:
gru_history, gru_pred, gru_mse, gru_mae, gru_time = train_and_evaluate(
    gru_model,
    X_train, y_train,
    X_test, y_test
)

print("GRU MSE:", round(gru_mse, 6))
print("GRU MAE:", round(gru_mae, 6))
print("Training time:", round(gru_time, 2), "seconds")


## 6. Compare the Three Models

The same dataset and sequence length are used for all models.


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Simple RNN", "LSTM", "GRU"],
    "MSE": [rnn_mse, lstm_mse, gru_mse],
    "MAE": [rnn_mae, lstm_mae, gru_mae],
    "Parameters": [
        rnn_model.count_params(),
        lstm_model.count_params(),
        gru_model.count_params()
    ],
    "Training Time (sec)": [
        rnn_time,
        lstm_time,
        gru_time
    ]
})

comparison


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test[:150], label="Actual")
plt.plot(rnn_pred[:150], label="RNN")
plt.plot(lstm_pred[:150], label="LSTM")
plt.plot(gru_pred[:150], label="GRU")
plt.title("RNN vs LSTM vs GRU")
plt.legend()
plt.show()


## 7. Stacked LSTM and `return_sequences=True`

When one LSTM layer is followed by another LSTM layer, the first layer must return a **sequence**, not only one final vector.

That is why we use:

```python
LSTM(50, return_sequences=True)
```


In [ ]:
stacked_lstm = Sequential([
    keras.Input(shape=(SEQ_LENGTH, 1)),
    LSTM(50, return_sequences=True),
    LSTM(30),
    Dense(1)
])

stacked_lstm.compile(optimizer="adam", loss="mse")
stacked_lstm.summary()


## 8. Predict the Next Value

Use the last sequence to predict the next point.


In [ ]:
last_sequence = series[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, 1)

print("RNN :", float(rnn_model.predict(last_sequence, verbose=0)[0, 0]))
print("LSTM:", float(lstm_model.predict(last_sequence, verbose=0)[0, 0]))
print("GRU :", float(gru_model.predict(last_sequence, verbose=0)[0, 0]))


# Student Tasks

Complete the following tasks independently.


## Task 1 — Understand the Input Shape

Suppose:

```python
X.shape = (500, 20, 3)
```

Answer:

1. How many samples are there?
2. How many timesteps are in each sample?
3. How many features exist at each timestep?
4. What does the second dimension represent?


## Task 2 — Change the Sequence Length

Repeat the LSTM experiment with:

- `seq_length = 5`
- `seq_length = 20`

For each experiment:

1. Create the sliding windows.
2. Build an LSTM with 32 units.
3. Train it.
4. Calculate test MSE.
5. Compare the results.

### Reflection
Does a longer sequence always improve performance? Why?


In [ ]:
# TODO — Task 2

# experiment_seq_length = 5
# experiment_seq_length = 20

# Create the new train/test sequences
# Build the LSTM
# Train
# Predict
# Calculate MSE


## Task 3 — Change the Number of Hidden Units

Train LSTM models with:

- 16 units
- 32 units
- 64 units

Create a comparison table containing:

- Hidden Units
- Number of Parameters
- Test MSE

### Question
Does increasing the number of units always improve the model?


In [ ]:
# TODO — Task 3

units_to_test = [16, 32, 64]

# Store your results in a list
# Then create a pandas DataFrame


## Task 4 — Build a Stacked LSTM

Build:

```text
LSTM(50)
LSTM(30)
Dense(1)
```

Make the architecture work correctly.

### Questions

1. Which LSTM layer needs `return_sequences=True`?
2. Why?
3. What happens if the first LSTM returns only one final vector?


In [ ]:
# TODO — Task 4

# stacked_model = Sequential([
#     keras.Input(shape=(SEQ_LENGTH, 1)),
#     ...
#     ...
#     Dense(1)
# ])


## Task 5 — Compare RNN, LSTM, and GRU

Using the comparison table:

1. Which model has the fewest parameters?
2. Which model trained fastest?
3. Which model achieved the lowest MSE?
4. Why should we not assume that one model is always best?


## Task 6 — Add Noise

Create a harder sequence:

```python
noisy_series = np.sin(x_axis) + np.random.normal(
    0, 0.15, size=len(x_axis)
)
```

Then:

1. Split the sequence chronologically.
2. Create sliding windows.
3. Train an LSTM.
4. Calculate test MSE.
5. Plot actual vs predicted values.

### Reflection
How does noise affect sequence learning?


In [ ]:
# TODO — Task 6

# noisy_series = np.sin(x_axis) + np.random.normal(
#     0, 0.15, size=len(x_axis)
# )

# Continue the full pipeline here.


## Bonus Task — Multiple Features

Create two features:

```python
feature_1 = np.sin(x_axis)
feature_2 = np.cos(x_axis)
```

Use both features to predict the next sine value.

Your input shape should become:

**(samples, timesteps, 2)**

### Questions

1. Why is the final dimension now `2`?
2. What must change in `input_shape`?
3. Does the whole model architecture need to change?


# Self-Check Questions

1. Why does a Dense network not naturally remember previous timesteps?
2. What is the role of the hidden state in an RNN?
3. What is the main limitation of a basic RNN on long sequences?
4. How does LSTM improve memory control?
5. What is the difference between cell state and hidden state?
6. Why is GRU considered simpler than LSTM?
7. What does `return_sequences=True` do?
8. Explain `(samples, timesteps, features)`.
9. What is the purpose of a sliding window?
10. Why should chronological sequence data not be randomly shuffled before splitting?


## Key Takeaways

- Sequence models are designed for data where **order matters**.
- RNN carries information using a **hidden state**.
- LSTM improves long-term memory using **gates and cell state**.
- GRU is a simpler gated recurrent architecture.
- Keras sequence models expect **3D input**:
  **samples × timesteps × features**
- Sliding windows convert a raw sequence into supervised learning samples.
